In [1]:
import numpy as np

from qiskit import transpile
from qiskit.circuit.library import QAOAAnsatz
from qiskit.transpiler.passes.routing.commuting_2q_gate_routing import SwapStrategy

from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as Sampler

from qiskit_ibm_runtime.fake_provider import FakeAlgiers, FakeGuadalupeV2

from qopt_best_practices.sat_mapping import SATMapper

from qiskit_qaoa.utils.circuit_graph_utils import circuit_to_graph, graph_to_operator, circuit_construction
from qiskit_qaoa.utils.hamiltonian_utils import get_Q_and_hamiltonian

In [2]:
filename = 'test_N2_W2'
p = 4
shots = 256
init_type = 'warm'

rng = np.random.default_rng()

data_file = f'/lustre/scratch127/qpg/jc59/out/oriented/qubo_data_{filename}.gfa.pkl'

Q, hamiltonian, offset = get_Q_and_hamiltonian(data_file)
qc = QAOAAnsatz(
    cost_operator=hamiltonian,
    reps = p,
    flatten=True
)
num_qubits = hamiltonian.num_qubits

backend = AerSimulator()
backend.set_option("n_qubits", num_qubits)

print(f'Backend: {backend}')
print(f'Num qubits in backend: {backend.configuration().to_dict()["n_qubits"]}')


transpiled_qc = transpile(qc, backend, optimization_level=3)


def print_circuit_info(qc, circuit_name):
    print(f'{circuit_name} has {qc.count_ops().get("cz", 0) + qc.count_ops().get("rzz", 0) + qc.count_ops().get("cx", 0) + qc.count_ops().get("swap", 0)+ qc.count_ops().get("ecr", 0)} 2Q gates \
    and {qc.depth(lambda instr: len(instr.qubits) > 1)} 2Q depth')


print_circuit_info(transpiled_qc, '(Transpiled) Circuit')

graph = circuit_to_graph(qc, qc.parameters[p])

swap_strat = SwapStrategy.from_line(range(graph.order()))
edge_coloring = {(idx, idx + 1): (idx + 1) % 2 for idx in range(graph.order())}

remapped_g, sat_map, min_sat_layers = SATMapper(timeout=30).remap_graph_with_sat(
    graph=graph, swap_strategy=swap_strat
)
if remapped_g is None:
    raise Exception('Failed to find initial layout')

cost_op = graph_to_operator(remapped_g)
singles = cost_op[cost_op.paulis.z.sum(axis=-1) == 1]
doubles = cost_op[cost_op.paulis.z.sum(axis=-1) == 2]

circ_dict = circuit_construction(singles, doubles, None, swap_strat, edge_coloring, {}, p)

circuit = circ_dict["circuit_to_sample"]
circuit.remove_final_measurements()
print_circuit_info(circuit, '(Transpiled) Remapped Circuit')


qaoa_depth = len(circuit.parameters) // 2


if init_type == 'ramp':
    t = 0.7 * p
    betas = np.linspace(
        (1 / p) * (t * (1 - 0.5 / p)), (1 / p) * (t * 0.5 / p), p
    )
    gammas = betas[::-1]
    init_params = betas.tolist() + gammas.tolist()
elif init_type == 'fixed':
    init_params = [0.439759390571553, 0.47523912591426465, 0.40683448432677694, 0.8665276969749175, 
                   0.07727284052271921, 0.25521259972136145, 0.04596182583685282, 0.04562939468439431]
    print('Using fixed init values')
elif init_type == 'warm':
    init_params = [ 9.33323444e-01,  7.08009649e-03,  7.36344025e-01,  9.37923754e-01,
        2.29973290e-02,  2.75044958e-03,  8.34971625e-04, -2.92071056e-04]
    print('Using warm init values')
else:
    init_params = rng.uniform(0, 1, qaoa_depth).tolist() + rng.uniform(0, 1, qaoa_depth).tolist()
print(f'Init: {init_params}')


def cvar(energies, alpha=1.0):
    sorted_energies = sorted(energies)
    end_idx = int(alpha * len(energies))
    return np.sum(sorted_energies[0:end_idx]) / end_idx





Backend: AerSimulator('aer_simulator')
Num qubits in backend: 10
(Transpiled) Circuit has 160 2Q gates     and 47 2Q depth
(Transpiled) Remapped Circuit has 496 2Q gates     and 112 2Q depth
Using warm init values
Init: [0.933323444, 0.00708009649, 0.736344025, 0.937923754, 0.022997329, 0.00275044958, 0.000834971625, -0.000292071056]


In [8]:

sampler = Sampler.from_backend(backend)
assigned_circuit = circuit.assign_parameters(init_params, inplace=False)
assigned_circuit.measure_all()
sampler_job = sampler.run([assigned_circuit], shots=shots)
sampler_result = sampler_job.result()


In [16]:
counts = sampler_result[0].data.meas.get_counts()
int_samples = [np.array([int(x) for x in sample[::-1]]) for sample in counts.keys()]
evals = np.array([
    sample @ Q @ sample for sample in int_samples
]) + offset
energies = [count * [evals[idx]] for idx, count in enumerate(counts.values())]
flat_energies = [x for xs in energies for x in xs]
total_energy = cvar(flat_energies, 0.25)


print(f'Energy: {total_energy}')
print(f'Best sample: {min(flat_energies)}')

Energy: 7.671875
Best sample: 0.0


In [17]:
backend_options = dict(
    method='statevector',
    device='CPU',
    memory=100000,
    precision='single'
)
backend = AerSimulator(**backend_options)

int_samples = [np.array([int(x) for x in np.binary_repr(y, width=hamiltonian.num_qubits)]) for y in range(2**hamiltonian.num_qubits)]
evals = np.array([
    sample @ Q @ sample for sample in int_samples
]) + offset





In [18]:
assigned_circuit = circuit.assign_parameters(init_params, inplace=False)
assigned_circuit.save_statevector()
sampler_job = backend.run([assigned_circuit])
sampler_result = sampler_job.result()


In [19]:
sampler_result.results[0]

ExperimentResult(shots=1024, success=True, meas_level=2, data=ExperimentResultData(statevector=Statevector([ 0.34033275-1.64864719e-01j,  0.16840157-1.52470559e-01j,
              0.08480063-1.53333336e-01j, ..., -0.00176804-9.11598618e-05j,
             -0.0020496 +1.54310802e-03j,  0.00184713+2.23361771e-04j],
            dims=(2, 2, 2, 2, 2, 2, 2, 2, 2, 2))), header={'creg_sizes': [], 'global_phase': 0.0, 'memory_slots': 0, 'n_qubits': 10, 'name': 'circuit-246-263', 'qreg_sizes': [['q', 10]], 'metadata': {}}, status=DONE, seed_simulator=3270679282, metadata={'time_taken': 0.004843185, 'num_bind_params': 1, 'parallel_state_update': 64, 'required_memory_mb': 1, 'input_qubit_map': [[9, 9], [8, 8], [7, 7], [6, 6], [5, 5], [4, 4], [3, 3], [2, 2], [1, 1], [0, 0]], 'method': 'statevector', 'device': 'CPU', 'num_qubits': 10, 'active_input_qubits': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9], 'num_clbits': 0, 'remapped_qubits': False, 'parallel_shots': 1, 'runtime_parameter_bind': False, 'max_memory_mb':

In [24]:
data = sampler_result.results[0].data
sv = np.asarray(data.statevector)

energy = np.sum(np.abs(sv) ** 2 * evals)

In [25]:
np.nonzero(sv)

(array([   0,    1,    2, ..., 1021, 1022, 1023], shape=(1024,)),)

In [26]:
evals

array([ 27.,  17.,  16., ..., 403., 443., 523.], shape=(1024,))

In [27]:
energy

np.float64(24.00711317356168)

In [ ]:
assigned_circuit.draw(fold=-1)